# 🌲 Clase 4 — Feature Engineering, Codificación y Selección de Variables

## Aplicaciones de Minería de Datos a Economía y Finanzas
### Maestría en Minería de Datos · UTN Facultad Regional Rosario · 2026

**Docente:** Dr. Darío Ezequiel Díaz · drdarioezequieldiaz@gmail.com

**Fecha:** Jueves 30 de abril de 2026

---

### Objetivos del notebook

1. **Construir** variables derivadas con sentido económico-financiero (ratios, indicadoras, interacciones, binning).
2. **Transformar** variables numéricas con los seis métodos canónicos (min-max, z-score, log, Box-Cox, Yeo-Johnson, quantile).
3. **Codificar** variables categóricas (one-hot, label, ordinal, target encoding regularizado, binary, frequency).
4. **Seleccionar** variables aplicando los tres paradigmas: filter, wrapper y embedded.
5. **Diagnosticar** multicolinealidad mediante el Factor de Inflación de Varianza (VIF) con eliminación iterativa.
6. **Integrar** todo el flujo en un Pipeline reproducible con `scikit-learn` y comparar el efecto sobre la performance predictiva.

### Dataset: German Credit Data (Statlog)

| Característica | Detalle |
|:---|:---|
| **Fuente** | UCI ML Repository / OpenML (id: `credit-g`) |
| **Registros** | 1.000 solicitantes de crédito |
| **Variables** | 20 predictoras + 1 objetivo |
| **Variable objetivo** | `class`: good (700) / bad (300) |
| **Desbalanceo** | 70/30 — moderado |
| **Dominio** | Evaluación de riesgo crediticio bancario (Alemania) |
| **Faltantes en el original** | 0% — punto de partida ideal para introducir condiciones controladas |

> **Hilo conductor pedagógico.** Trabajaremos sobre el mismo dataset que veníamos preparando en la Clase 3. Allí lo dejamos limpio. Aquí lo enriquecemos con features derivadas, lo transformamos con criterio y lo reducimos a un subconjunto óptimo de variables para alimentar el primer modelo de scoring crediticio que entrenaremos en la Clase 5.


---
## 0. Configuración del entorno

Cargamos las librerías necesarias. Algunas no vienen preinstaladas en Colab y requieren instalación previa: `category_encoders` (codificadores avanzados) y `scikit-learn` actualizado para acceder a `mutual_info_classif`, `RFECV` y `PowerTransformer`.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 0.1 INSTALACIÓN DE PAQUETES (descomentar si es la primera ejecución en Colab)
# ══════════════════════════════════════════════════════════════
# !pip install category_encoders --quiet
# !pip install statsmodels --quiet --upgrade

print('[OK] Paquetes verificados — proceder a importar')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 0.2 IMPORTS
# ══════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

# scikit-learn — preprocesamiento
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import (
    MinMaxScaler, StandardScaler, RobustScaler,
    PowerTransformer, QuantileTransformer,
    OneHotEncoder, OrdinalEncoder, LabelEncoder
)

# scikit-learn — selección de variables
from sklearn.feature_selection import (
    mutual_info_classif, chi2, f_classif,
    RFE, RFECV, SelectFromModel
)

# scikit-learn — modelos
from sklearn.linear_model import LogisticRegression, Lasso, LassoCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score, classification_report

# Codificadores avanzados
import category_encoders as ce

# Multicolinealidad
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Configuración estética
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.0)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
warnings.filterwarnings('ignore')

# Paleta Forest & Moss para los gráficos
PALETA = {
    'forest':  '#1F4023',
    'moss':    '#4A7C59',
    'mosslt':  '#8FB996',
    'teal':    '#2A8775',
    'gold':    '#D4A017',
    'rust':    '#B83A4A',
    'plum':    '#6B4AA6',
}

# Reproducibilidad
SEED = 42
np.random.seed(SEED)

print('[OK] Librerias importadas correctamente')
print(f'    pandas       : {pd.__version__}')
print(f'    numpy        : {np.__version__}')
print(f'    seaborn      : {sns.__version__}')
print(f'    category_enc : {ce.__version__}')

---
## 1. Carga e inspección del German Credit Dataset

El dataset se descarga desde OpenML mediante la función `fetch_openml` de scikit-learn. Es la práctica más reproducible: evita archivos locales y garantiza que cualquier maestrando pueda ejecutar el notebook sin configuraciones adicionales.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 1.1 CARGA DEL DATASET
# ══════════════════════════════════════════════════════════════

print('Descargando German Credit Dataset desde OpenML...')
credit = fetch_openml('credit-g', version=1, as_frame=True, parser='auto')
df = credit.frame.copy()

# Renombrado de la columna objetivo para mayor claridad
df = df.rename(columns={'class': 'target'})

# Mapeo binario explicito: bad=1 (default), good=0 (al dia)
# Esta convencion sigue la practica del scoring crediticio donde
# el evento de interes (default) se codifica como 1.
df['target_bin'] = (df['target'] == 'bad').astype(int)

print(f'\n[OK] Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'    Tasa de default (bad): {df.target_bin.mean():.1%}')
print(f'    Tasa de buen pago (good): {1 - df.target_bin.mean():.1%}')

df.head()

In [ ]:
# ══════════════════════════════════════════════════════════════
# 1.2 INSPECCION DE TIPOS Y CARDINALIDADES
# ══════════════════════════════════════════════════════════════

resumen = pd.DataFrame({
    'tipo': df.dtypes.astype(str),
    'n_unicos': df.nunique(),
    'n_faltantes': df.isnull().sum(),
    'pct_faltantes': (df.isnull().mean() * 100).round(2)
})
resumen['categoria'] = np.where(
    resumen['tipo'].str.contains('category|object'),
    'categorica', 'numerica'
)

print('Resumen estructural del dataset:')
print('=' * 70)
print(resumen.to_string())
print('=' * 70)
print(f'\nVariables numericas: {(resumen.categoria == "numerica").sum()}')
print(f'Variables categoricas: {(resumen.categoria == "categorica").sum()}')
print(f'Total faltantes: {df.isnull().sum().sum()}  (dataset limpio de origen)')

### 📝 Interpretación

El German Credit Dataset original no presenta faltantes —es la propiedad que aprovechamos para introducir condiciones controladas en clases anteriores. Tenemos una mezcla equilibrada entre variables numéricas (las cantitativas como `credit_amount`, `duration`, `age`) y categóricas (los atributos cualitativos como `purpose`, `housing`, `job`, además de las codificaciones ordinales de bancos alemanes como `checking_status`).

La tasa de default del 30% genera un desbalanceo moderado, no severo. Esto significa que podemos trabajar con modelos sin estrategias agresivas de remuestreo —el SMOTE y los costos asimétricos los abordamos formalmente en la Clase 6. Para esta sesión de feature engineering, el desbalanceo no afecta materialmente nuestro pipeline.

---
## 2. Partición train / test antes de cualquier transformación

**Regla de oro absoluta del feature engineering:** toda transformación que aprende parámetros desde los datos —media, desvío, percentiles, codificaciones, etc.— debe aprenderlos exclusivamente sobre el conjunto de entrenamiento. Aplicar `fit` sobre el dataset completo antes de partir produce *data leakage por preprocesamiento*: el modelo verá información del test indirectamente y la performance reportada será optimista.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 2.1 PARTICION ESTRATIFICADA TRAIN/TEST
# ══════════════════════════════════════════════════════════════

X = df.drop(columns=['target', 'target_bin'])
y = df['target_bin']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=SEED
)

print('PARTICION ESTRATIFICADA')
print('=' * 60)
print(f'Train: {X_train.shape[0]} obs  |  Tasa default: {y_train.mean():.1%}')
print(f'Test:  {X_test.shape[0]} obs  |  Tasa default: {y_test.mean():.1%}')
print('=' * 60)
print('\nLa estratificacion preserva la proporcion de clases en ambos splits.')
print('Esto es esencial cuando hay desbalanceo: una particion aleatoria pura')
print('podria producir un test con tasa muy distinta y sesgar las metricas.')

---
# 🧱 BLOQUE 1 — Feature Engineering con sentido económico

> *"Coming up with features is difficult, time-consuming, requires expert knowledge. Applied machine learning is basically feature engineering."* — **Andrew Ng**

El feature engineering refiere al proceso creativo de construir nuevas variables derivadas a partir de los datos crudos, con el propósito de capturar información predictiva que las variables originales no expresan directamente.

**Tres dimensiones lo definen:**

1. **Conocimiento de dominio**: traducir hipótesis económicas en variables medibles.
2. **Operación matemática**: razones, diferencias, productos, agregaciones, codificaciones.
3. **Validación empírica**: verificar que la feature efectivamente aporte poder predictivo.

> **Cada feature derivada es una hipótesis económica codificada en el dataset.**

A continuación construiremos features pertenecientes a las **cinco familias canónicas** aplicables al German Credit Dataset.

---
## 1.1 Familia 1 — Ratios financieros

Los ratios neutralizan el efecto de escala y permiten comparar observaciones heterogéneas. En el German Credit, los más relevantes surgen de combinar el monto solicitado, el plazo y la antigüedad laboral.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 1.1 RATIOS FINANCIEROS
# ══════════════════════════════════════════════════════════════
# Trabajamos sobre copias del train y test para no contaminar los originales.

X_train_fe = X_train.copy()
X_test_fe  = X_test.copy()

# Ratio 1: pago mensual implicito (monto / plazo)
# Hipotesis economica: la capacidad de pago mensual predice mejor que el monto absoluto.
X_train_fe['monthly_payment'] = X_train_fe['credit_amount'] / X_train_fe['duration']
X_test_fe['monthly_payment']  = X_test_fe['credit_amount']  / X_test_fe['duration']

# Ratio 2: monto por anio de antiguedad laboral (proxy de capacidad de respaldo)
# Para evitar division por cero, sumamos 1 (las categorias 'unemployed' / '<1' valen ~0)
# Convertimos employment a numerico ordinal previamente
emp_map = {
    'unemployed': 0,
    '<1': 0.5,
    '1<=X<4': 2.5,
    '4<=X<7': 5.5,
    '>=7': 10.0
}
X_train_fe['employment_num'] = X_train_fe['employment'].astype(str).map(emp_map)
X_test_fe['employment_num']  = X_test_fe['employment'].astype(str).map(emp_map)

X_train_fe['credit_per_emp_year'] = X_train_fe['credit_amount'] / (X_train_fe['employment_num'] + 1)
X_test_fe['credit_per_emp_year']  = X_test_fe['credit_amount']  / (X_test_fe['employment_num']  + 1)

# Ratio 3: amount-to-age (proxy de exposicion relativa al ciclo de vida)
X_train_fe['amount_per_age'] = X_train_fe['credit_amount'] / X_train_fe['age']
X_test_fe['amount_per_age']  = X_test_fe['credit_amount']  / X_test_fe['age']

print('Ratios construidos:')
print('  monthly_payment      = credit_amount / duration')
print('  credit_per_emp_year  = credit_amount / (employment_num + 1)')
print('  amount_per_age       = credit_amount / age')
print()

ratios = ['monthly_payment', 'credit_per_emp_year', 'amount_per_age']
print('Estadisticas descriptivas en TRAIN:')
print(X_train_fe[ratios].describe().round(1))

In [ ]:
# ══════════════════════════════════════════════════════════════
# 1.1.b VALIDACION BIVARIADA: poder discriminante de los ratios
# ══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, var in zip(axes, ratios):
    # Boxplot por clase
    df_plot = pd.DataFrame({var: X_train_fe[var], 'target': y_train.map({0:'good', 1:'bad'})})
    sns.boxplot(data=df_plot, x='target', y=var, ax=ax,
                palette={'good': PALETA['moss'], 'bad': PALETA['rust']})
    ax.set_title(f'{var}\npor clase de riesgo', fontweight='bold', fontsize=11)
    ax.set_xlabel('')
    # Test de Mann-Whitney para validar diferencia
    stat, pval = stats.mannwhitneyu(
        X_train_fe.loc[y_train==0, var],
        X_train_fe.loc[y_train==1, var]
    )
    ax.text(0.5, 0.95, f'Mann-Whitney p = {pval:.4f}',
            transform=ax.transAxes, ha='center', va='top',
            fontsize=10, color=PALETA['forest'], fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=PALETA['gold']))

plt.suptitle('Validacion bivariada de ratios financieros vs target', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nLectura: si el p-valor de Mann-Whitney es < 0.05, las distribuciones')
print('por clase difieren significativamente y el ratio aporta poder discriminante.')

### 📝 Interpretación

Los tres ratios capturan dimensiones complementarias del riesgo crediticio. **`monthly_payment`** sintetiza el esfuerzo financiero mensual del solicitante; valores altos sugieren mayor presión de caja y suelen asociarse a mayor riesgo de default. **`credit_per_emp_year`** introduce la antigüedad laboral como denominador: solicitantes con poca trayectoria que piden montos elevados quedan flagueados con valores extremos. **`amount_per_age`** opera de manera análoga sobre el ciclo de vida.

El test de Mann-Whitney es la elección apropiada porque no asume normalidad de las distribuciones —las variables monetarias del German Credit tienen asimetría positiva pronunciada. Cuando el p-valor cae por debajo de 0.05, podemos rechazar la hipótesis nula de igualdad de distribuciones entre buenos y malos pagadores: el ratio efectivamente discrimina.

---
## 1.2 Familia 2 — Variables indicadoras (binning binario)

Las indicadoras son la forma más simple de discretización. Codifican una hipótesis dicotómica del dominio: *¿el solicitante pertenece o no a un grupo de riesgo identificable?* Aunque pierden granularidad, ganan interpretabilidad y robustez ante outliers.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 1.2 INDICADORAS
# ══════════════════════════════════════════════════════════════

# Indicadora 1: solicitante joven (< 25 anios)
# Hipotesis: los menores de 25 tienen menor estabilidad financiera y mayor riesgo.
X_train_fe['is_young_borrower'] = (X_train_fe['age'] < 25).astype(int)
X_test_fe['is_young_borrower']  = (X_test_fe['age']  < 25).astype(int)

# Indicadora 2: empleo estable (>= 4 anios)
# Hipotesis: empleos prolongados sugieren ingresos predecibles.
X_train_fe['stable_employment'] = (X_train_fe['employment_num'] >= 4).astype(int)
X_test_fe['stable_employment']  = (X_test_fe['employment_num']  >= 4).astype(int)

# Indicadora 3: prestamo de larga duracion (> 24 meses)
# Hipotesis: plazos largos exponen mas tiempo al riesgo y a shocks economicos.
X_train_fe['long_term_loan'] = (X_train_fe['duration'] > 24).astype(int)
X_test_fe['long_term_loan']  = (X_test_fe['duration']  > 24).astype(int)

# Indicadora 4: monto alto (cuartil superior del train)
q75_amount = X_train_fe['credit_amount'].quantile(0.75)
X_train_fe['high_amount'] = (X_train_fe['credit_amount'] > q75_amount).astype(int)
X_test_fe['high_amount']  = (X_test_fe['credit_amount']  > q75_amount).astype(int)

print(f'Umbral monto alto (Q75 del train): {q75_amount:.0f} DM')
print()

# Tasa de default por categoria de indicadora
indicadoras = ['is_young_borrower', 'stable_employment', 'long_term_loan', 'high_amount']

print('Tasa de default condicionada por cada indicadora:')
print('=' * 75)
print(f'{"Indicadora":<22} {"=0":<12} {"=1":<12} {"Diferencia":<12} {"WoE":<8}')
print('-' * 75)

# Weight of Evidence simplificado: log(odds_1 / odds_0)
for ind in indicadoras:
    tasa_0 = y_train[X_train_fe[ind] == 0].mean()
    tasa_1 = y_train[X_train_fe[ind] == 1].mean()
    diff   = tasa_1 - tasa_0
    # WoE = log(P(bad|x=1)/P(good|x=1)) - log(P(bad|x=0)/P(good|x=0))
    odds_1 = tasa_1 / max(1 - tasa_1, 1e-6)
    odds_0 = tasa_0 / max(1 - tasa_0, 1e-6)
    woe = np.log(odds_1 / odds_0) if odds_0 > 0 else np.nan
    print(f'{ind:<22} {tasa_0:>6.1%}      {tasa_1:>6.1%}      {diff:>+7.1%}     {woe:>+5.2f}')
print('=' * 75)
print('\nWoE interpretacion: |WoE| > 0.5 indica poder discriminante notorio.')
print('Signo positivo: la categoria 1 concentra mas defaults que la categoria 0.')

### 📝 Interpretación

El **Weight of Evidence (WoE)** es la métrica canónica del scoring crediticio para evaluar el poder discriminante de una variable categórica respecto del target. Surge de la teoría de Bayes: cuantifica cuánto la observación de la categoría modifica las odds de default.

- $|WoE| > 0.5$: la categoría tiene poder discriminante notable.
- $WoE > 0$: la categoría 1 concentra más defaults que la categoría 0.
- $WoE < 0$: la categoría 1 concentra menos defaults (es protectora).

Las indicadoras con WoE positivo y magnitud considerable son candidatas firmes para el modelo final. Las de WoE cercano a cero pueden descartarse en la selección.

---
## 1.3 Familia 3 — Interacciones multiplicativas

Las interacciones capturan efectos no lineales que los modelos lineales no detectan por sí solos. La hipótesis económica detrás es la *solvencia compuesta*: ciertas combinaciones de atributos producen perfiles de riesgo distintos a los que cada atributo produce por separado.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 1.3 INTERACCIONES MULTIPLICATIVAS
# ══════════════════════════════════════════════════════════════

# Interaccion 1: antiguedad x monto (solvencia compuesta)
X_train_fe['emp_x_amount'] = X_train_fe['employment_num'] * X_train_fe['credit_amount']
X_test_fe['emp_x_amount']  = X_test_fe['employment_num']  * X_test_fe['credit_amount']

# Interaccion 2: edad x duracion (exposicion temporal acumulada)
X_train_fe['age_x_duration'] = X_train_fe['age'] * X_train_fe['duration']
X_test_fe['age_x_duration']  = X_test_fe['age']  * X_test_fe['duration']

# Interaccion 3: indicadora compuesta (joven Y prestamo largo) - efecto multiplicativo
X_train_fe['young_long_loan'] = X_train_fe['is_young_borrower'] * X_train_fe['long_term_loan']
X_test_fe['young_long_loan']  = X_test_fe['is_young_borrower']  * X_test_fe['long_term_loan']

interacciones = ['emp_x_amount', 'age_x_duration', 'young_long_loan']

# Correlacion de Spearman con el target (robusto a no linealidad)
print('Correlacion de Spearman entre interacciones y target:')
print('=' * 60)
for var in interacciones:
    rho, pval = stats.spearmanr(X_train_fe[var], y_train)
    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ' '
    print(f'  {var:<20} rho = {rho:+.4f}  p = {pval:.4f}  {sig}')
print('=' * 60)

# Caso interesante: la interaccion young_long_loan
# Comparemos su tasa de default contra las componentes individuales
print('\nCaso especial: interseccion young_long_loan')
print('-' * 60)
print(f'  Solo joven (no prestamo largo):    {y_train[(X_train_fe.is_young_borrower==1)&(X_train_fe.long_term_loan==0)].mean():.1%}')
print(f'  Solo prestamo largo (no joven):    {y_train[(X_train_fe.is_young_borrower==0)&(X_train_fe.long_term_loan==1)].mean():.1%}')
print(f'  Ambas condiciones (interaccion):   {y_train[X_train_fe.young_long_loan==1].mean():.1%}')
print(f'  Ninguna de las dos (baseline):     {y_train[(X_train_fe.is_young_borrower==0)&(X_train_fe.long_term_loan==0)].mean():.1%}')

### 📝 Interpretación

El caso de la interacción `young_long_loan` ilustra de forma cristalina por qué las interacciones importan en modelos lineales. La regresión logística supone que el efecto de cada variable es aditivo: si $\beta_1$ captura el efecto de ser joven y $\beta_2$ el efecto de tomar un préstamo largo, el modelo predice que un solicitante joven con préstamo largo tiene un riesgo igual a la suma de ambos efectos. Si la realidad muestra un efecto sinérgico —el riesgo combinado es mayor que la suma—, el modelo no podrá capturarlo a menos que le demos explícitamente la interacción $X_1 \cdot X_2$.

Los modelos basados en árboles (Random Forest, XGBoost) capturan interacciones automáticamente mediante splits sucesivos. Por eso, en proyectos donde el algoritmo final será de boosting, la construcción manual de interacciones aporta menos. En modelos lineales para scoring regulatorio, en cambio, son indispensables.

---
## 1.4 Familia 4 — Discretización por tramos (binning multiclase)

A diferencia de las indicadoras binarias, el binning multiclase divide la variable continua en múltiples tramos —típicamente cuartiles, quintiles o cortes definidos por el negocio. Es la familia más usada en scoring crediticio bancario porque facilita la construcción de scorecards interpretables.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 1.4 BINNING MULTICLASE
# ══════════════════════════════════════════════════════════════

# Bin 1: edad por tramos de ciclo de vida (definidos por negocio)
edad_bins = [0, 25, 35, 50, 100]
edad_labels = ['joven', 'adulto_joven', 'adulto_medio', 'mayor']
X_train_fe['age_bin'] = pd.cut(X_train_fe['age'], bins=edad_bins, labels=edad_labels)
X_test_fe['age_bin']  = pd.cut(X_test_fe['age'],  bins=edad_bins, labels=edad_labels)

# Bin 2: duracion por cuartiles (data-driven sobre TRAIN)
duracion_quartiles = X_train_fe['duration'].quantile([0.25, 0.5, 0.75]).values
duracion_bins = [0] + list(duracion_quartiles) + [np.inf]
X_train_fe['duration_q'] = pd.cut(X_train_fe['duration'], bins=duracion_bins,
                                   labels=['Q1', 'Q2', 'Q3', 'Q4'])
X_test_fe['duration_q']  = pd.cut(X_test_fe['duration'],  bins=duracion_bins,
                                   labels=['Q1', 'Q2', 'Q3', 'Q4'])

print(f'Cortes de duracion (cuartiles del train): {duracion_quartiles.tolist()}')
print()

# Visualizacion: tasa de default por bin
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

for ax, var, titulo in zip(axes, ['age_bin', 'duration_q'],
                            ['Edad por ciclo de vida', 'Duracion por cuartiles']):
    tasa = y_train.groupby(X_train_fe[var]).mean().sort_index()
    n_obs = X_train_fe[var].value_counts().sort_index()
    bars = ax.bar(tasa.index.astype(str), tasa.values,
                  color=PALETA['moss'], edgecolor=PALETA['forest'], linewidth=1.2)
    ax.axhline(y_train.mean(), color=PALETA['rust'], linestyle='--',
               linewidth=1.5, label=f'Tasa global = {y_train.mean():.1%}')
    for bar, n in zip(bars, n_obs.values):
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
                f'{h:.1%}\n(n={n})', ha='center', va='bottom', fontsize=9)
    ax.set_title(titulo, fontweight='bold')
    ax.set_ylabel('Tasa de default')
    ax.set_ylim(0, max(tasa.values) * 1.25)
    ax.legend()

plt.suptitle('Tasa de default por bin', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 📝 Interpretación

El binning multiclase permite detectar **relaciones no monotónicas** entre la variable y el target —patrones que un modelo lineal sobre la variable original no captura. En el caso de la edad, observamos típicamente una forma de U invertida invertida: los muy jóvenes y los más mayores tienden a presentar mayores tasas de default que los grupos centrales. Sin binning, una regresión logística asignaría un coeficiente único que no puede capturar esta forma.

En la duración del crédito, en cambio, suele observarse monotonicidad creciente: cuanto más largo el plazo, mayor el riesgo, por la mayor exposición temporal a shocks. En esos casos el binning es opcional —la variable continua ya captura la información—.

> **Decisión metodológica.** Cuando la relación es monotónica, dejar la variable continua. Cuando es no monotónica (forma de U, J, etc.), binnear. La inspección visual del WoE por bin es la herramienta diagnóstica.

---
## 1.5 Familia 5 — Variables ordinales agregadas (liquidity score)

La última familia construye un *score compuesto* sumando codificaciones ordinales de variables relacionadas conceptualmente. Es una técnica que reduce dimensionalidad y resuelve multicolinealidad en una sola operación.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 1.5 SCORE DE LIQUIDEZ: agregacion ordinal
# ══════════════════════════════════════════════════════════════

# Mapeos ordinales fundados en la jerarquia natural de los niveles
# checking_status: estado de la cuenta corriente (mayor saldo = menor riesgo)
checking_map = {
    'no checking': 0,        # sin cuenta - peor situacion
    '<0': 1,                 # sobregirado
    '0<=X<200': 2,           # saldo bajo
    '>=200': 3               # saldo alto - mejor situacion
}

# savings_status: nivel de ahorros (mayor ahorro = menor riesgo)
savings_map = {
    'no known savings': 0,
    '<100': 1,
    '100<=X<500': 2,
    '500<=X<1000': 3,
    '>=1000': 4
}

# Aplicacion del mapeo
X_train_fe['checking_ord'] = X_train_fe['checking_status'].astype(str).map(checking_map)
X_train_fe['savings_ord']  = X_train_fe['savings_status'].astype(str).map(savings_map)
X_test_fe['checking_ord']  = X_test_fe['checking_status'].astype(str).map(checking_map)
X_test_fe['savings_ord']   = X_test_fe['savings_status'].astype(str).map(savings_map)

# Score de liquidez = suma de las dos ordinalizaciones
X_train_fe['liquidity_score'] = X_train_fe['checking_ord'] + X_train_fe['savings_ord']
X_test_fe['liquidity_score']  = X_test_fe['checking_ord']  + X_test_fe['savings_ord']

# Tasa de default por nivel de liquidity_score
print('Tasa de default por nivel de liquidity_score:')
print('=' * 55)
print(f'{"Score":<8} {"N obs":<8} {"Tasa default":<14} {"Var. vs prev":<12}')
print('-' * 55)
prev_tasa = None
for score in sorted(X_train_fe['liquidity_score'].unique()):
    mask = X_train_fe['liquidity_score'] == score
    n = mask.sum()
    tasa = y_train[mask].mean()
    var = '' if prev_tasa is None else f'{tasa - prev_tasa:+.2%}'
    print(f'{score:<8} {n:<8} {tasa:<14.1%} {var:<12}')
    prev_tasa = tasa
print('=' * 55)

# Correlacion de Spearman con target
rho, pval = stats.spearmanr(X_train_fe['liquidity_score'], y_train)
print(f'\nSpearman liquidity_score vs target: rho = {rho:+.4f}, p = {pval:.4e}')
print('Interpretacion: rho negativo confirma que mayor liquidez -> menor probabilidad de default.')

### 📝 Interpretación

El `liquidity_score` consolida en una sola dimensión la información de dos variables conceptualmente relacionadas (cuenta corriente y ahorros). Reducir dos variables correlacionadas a una composición ordinal cumple **dos objetivos simultáneamente**: (1) preserva la información económica de respaldo financiero del solicitante, y (2) elimina la multicolinealidad entre `checking_ord` y `savings_ord` —porque el modelo final solo verá la suma, no las componentes individuales.

La correlación de Spearman negativa con el target confirma la hipótesis económica: a mayor reserva de liquidez, menor probabilidad de default. El test de significancia con p-valor extremadamente bajo respalda que la relación no es producto del azar muestral.

---
## 1.6 Síntesis del Bloque 1 — Inventario de features construidas


In [ ]:
# ══════════════════════════════════════════════════════════════
# 1.6 INVENTARIO Y RANKING POR INFORMACION MUTUA
# ══════════════════════════════════════════════════════════════

features_creadas = {
    'Ratios':        ['monthly_payment', 'credit_per_emp_year', 'amount_per_age'],
    'Indicadoras':   ['is_young_borrower', 'stable_employment', 'long_term_loan', 'high_amount'],
    'Interacciones': ['emp_x_amount', 'age_x_duration', 'young_long_loan'],
    'Binning':       ['age_bin', 'duration_q'],
    'Ordinales':     ['checking_ord', 'savings_ord', 'liquidity_score']
}

print('=' * 65)
print('INVENTARIO DE FEATURES CONSTRUIDAS EN EL BLOQUE 1')
print('=' * 65)
total = 0
for familia, vars in features_creadas.items():
    print(f'\n{familia} ({len(vars)} features):')
    for v in vars:
        print(f'   - {v}')
    total += len(vars)
print(f'\n{"=" * 65}')
print(f'TOTAL: {total} features derivadas')
print('=' * 65)

# Ranking por informacion mutua sobre las features NUMERICAS construidas
features_num = [v for fam in features_creadas.values() for v in fam
                if v not in ['age_bin', 'duration_q']]
mi = mutual_info_classif(X_train_fe[features_num], y_train, random_state=SEED)
mi_df = pd.DataFrame({'feature': features_num, 'mutual_info': mi}).sort_values('mutual_info', ascending=False)

print('\nRanking por informacion mutua (IM) con el target:')
print('-' * 50)
for _, row in mi_df.iterrows():
    bar = '█' * int(row['mutual_info'] * 100)
    print(f'  {row["feature"]:<22} {row["mutual_info"]:.4f}  {bar}')

### 📝 Síntesis del Bloque 1

Construimos **15 features derivadas** organizadas en cinco familias canónicas. Cada una codifica una hipótesis económica explícita sobre el riesgo crediticio. El ranking por información mutua nos da una primera lectura preliminar del poder predictivo: las features con mayor IM son candidatas firmes a sobrevivir el proceso de selección que abordaremos en el Bloque 4.

> **Observación profesional.** El feature engineering bien hecho típicamente eleva el AUC de un modelo de scoring entre 3 y 8 puntos respecto del baseline con variables crudas. En la Sección 6 cuantificaremos esta mejora empíricamente sobre el German Credit.


---
# 🔄 BLOQUE 2 — Transformación de variables numéricas

La mayoría de los algoritmos de machine learning tienen supuestos sobre la escala o la distribución de las entradas que rara vez se cumplen automáticamente con datos económicos crudos. La regresión logística regularizada penaliza coeficientes proporcionalmente a su magnitud, las redes neuronales saturan sus activaciones ante valores extremos, los algoritmos basados en distancias se ven dominados por las variables de mayor magnitud.

Recorreremos las **seis transformaciones canónicas** sobre la variable `credit_amount` —típica candidata a transformación por su asimetría positiva pronunciada— para visualizar el efecto de cada una y construir el criterio decisional.

| Transformación | Fórmula | Caso de uso |
|:---|:---|:---|
| **Min-Max** | $x' = (x - x_{min}) / (x_{max} - x_{min})$ | Rango acotado, redes neuronales |
| **Z-score** | $z = (x - \mu) / \sigma$ | Modelos con regularización, distancias |
| **Logarítmica** | $x' = \log(1 + x)$ | Variables monetarias, asimetría positiva |
| **Box-Cox** | $x' = (x^\lambda - 1)/\lambda$ | Aproximar normalidad, $x > 0$ |
| **Yeo-Johnson** | Box-Cox generalizada | Aproximar normalidad, $x \in \mathbb{R}$ |
| **Quantile** | Mapeo a percentiles | Robustez extrema a outliers |

> **Regla de oro.** Toda transformación se ajusta exclusivamente sobre el conjunto de entrenamiento. El `fit` aprende los parámetros (mínimo, máximo, media, desvío, $\lambda$). El `transform` los aplica. Sobre el test solo se invoca `transform` con los parámetros aprendidos en train.

---
## 2.1 Diagnóstico previo: ¿qué tan asimétrica es `credit_amount`?


In [ ]:
# ══════════════════════════════════════════════════════════════
# 2.1 DIAGNOSTICO DE ASIMETRIA Y CURTOSIS
# ══════════════════════════════════════════════════════════════

variable = 'credit_amount'
serie = X_train_fe[variable]

skew_val = stats.skew(serie)
kurt_val = stats.kurtosis(serie)
shapiro_stat, shapiro_p = stats.shapiro(serie.sample(min(500, len(serie)), random_state=SEED))

print(f'Diagnostico distribucional de {variable}:')
print('=' * 55)
print(f'  N obs           : {len(serie)}')
print(f'  Media           : {serie.mean():>10.0f} DM')
print(f'  Mediana         : {serie.median():>10.0f} DM')
print(f'  Desvio          : {serie.std():>10.0f} DM')
print(f'  Min - Max       : {serie.min():.0f} - {serie.max():.0f} DM')
print(f'  Asimetria (skew): {skew_val:>10.3f}    [normal ~ 0]')
print(f'  Curtosis        : {kurt_val:>10.3f}    [normal ~ 0]')
print(f'  Shapiro-Wilk p  : {shapiro_p:>10.4e}  [normalidad rechazada si p < 0.05]')
print('=' * 55)
print(f'\nDiagnostico: asimetria positiva pronunciada (skew = {skew_val:.2f}).')
print('La variable es candidata clara a transformacion logaritmica o Box-Cox.')

### 📝 Interpretación

La asimetría (skewness) cuantifica la desviación respecto de la simetría: valores positivos indican cola derecha extendida (típico de variables monetarias), negativos cola izquierda. Una asimetría superior a 1 en magnitud suele requerir transformación.

La curtosis mide la concentración de masa en las colas: valores altos sugieren presencia de eventos extremos. El test de Shapiro-Wilk formaliza el diagnóstico de normalidad: con p-valor por debajo de 0.05 rechazamos la normalidad, lo cual es prácticamente la regla en variables económicas crudas.

---
## 2.2 Aplicación comparativa de las seis transformaciones


In [ ]:
# ══════════════════════════════════════════════════════════════
# 2.2 APLICACION DE LAS 6 TRANSFORMACIONES SOBRE CREDIT_AMOUNT
# ══════════════════════════════════════════════════════════════

# Importante: cada transformacion se ajusta sobre TRAIN y se aplica sobre TEST.
# Trabajamos con la columna como matriz 2D (requerido por sklearn).

x_train = X_train_fe[[variable]].values
x_test  = X_test_fe[[variable]].values

# Diccionario de transformaciones
transformaciones = {}

# (1) Min-Max
mm = MinMaxScaler()
transformaciones['minmax'] = (mm.fit_transform(x_train).ravel(), mm.transform(x_test).ravel())

# (2) Z-score
zs = StandardScaler()
transformaciones['zscore'] = (zs.fit_transform(x_train).ravel(), zs.transform(x_test).ravel())

# (3) Logaritmica - log1p (sin sklearn, fit no requiere parametros aprendidos)
transformaciones['log'] = (np.log1p(x_train).ravel(), np.log1p(x_test).ravel())

# (4) Box-Cox: requiere x > 0 (cumple, credit_amount es > 0)
bc = PowerTransformer(method='box-cox', standardize=False)
transformaciones['boxcox'] = (bc.fit_transform(x_train).ravel(), bc.transform(x_test).ravel())
lambda_bc = bc.lambdas_[0]

# (5) Yeo-Johnson: admite valores no positivos
yj = PowerTransformer(method='yeo-johnson', standardize=False)
transformaciones['yeojohnson'] = (yj.fit_transform(x_train).ravel(), yj.transform(x_test).ravel())
lambda_yj = yj.lambdas_[0]

# (6) Quantile transform a normal
qt = QuantileTransformer(output_distribution='normal', random_state=SEED, n_quantiles=min(500, len(x_train)))
transformaciones['quantile'] = (qt.fit_transform(x_train).ravel(), qt.transform(x_test).ravel())

print('Transformaciones ajustadas sobre TRAIN y aplicadas a TEST:')
print('=' * 55)
print(f'  Box-Cox lambda     : {lambda_bc:.4f}')
print(f'  Yeo-Johnson lambda : {lambda_yj:.4f}')
print(f'  Quantile output    : normal estandar')
print('=' * 55)

# Tabla comparativa de asimetria post-transformacion
print('\nAsimetria de la variable transformada (TRAIN):')
print('-' * 50)
print(f'  {"Original":<20} {skew_val:>+8.4f}')
for nombre, (tr_train, _) in transformaciones.items():
    s = stats.skew(tr_train)
    print(f'  {nombre:<20} {s:>+8.4f}')
print('-' * 50)
print('\nObjetivo: skew cercana a 0 (distribucion simetrica/normal).')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 2.3 VISUALIZACION COMPARATIVA: 6 transformaciones + original
# ══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.ravel()

# Original
axes[0].hist(x_train.ravel(), bins=40, color=PALETA['rust'], edgecolor='white', alpha=0.85)
axes[0].set_title(f'ORIGINAL\nskew = {skew_val:+.3f}', fontweight='bold')
axes[0].set_xlabel(variable)
axes[0].set_ylabel('Frecuencia')

# Las 6 transformaciones
colores = [PALETA['forest'], PALETA['teal'], PALETA['gold'],
           PALETA['plum'], PALETA['moss'], PALETA['mosslt']]
nombres_pretty = {
    'minmax':     'MIN-MAX [0,1]',
    'zscore':     'Z-SCORE',
    'log':        'LOG(1+x)',
    'boxcox':     f'BOX-COX (λ={lambda_bc:.2f})',
    'yeojohnson': f'YEO-JOHNSON (λ={lambda_yj:.2f})',
    'quantile':   'QUANTILE -> NORMAL'
}

for i, (clave, color) in enumerate(zip(transformaciones, colores), 1):
    tr_train, _ = transformaciones[clave]
    s = stats.skew(tr_train)
    axes[i].hist(tr_train, bins=40, color=color, edgecolor='white', alpha=0.85)
    axes[i].set_title(f'{nombres_pretty[clave]}\nskew = {s:+.3f}', fontweight='bold')
    axes[i].set_xlabel('valor transformado')
    axes[i].set_ylabel('Frecuencia')

# Diagnostico Q-Q plots de las 3 mejores aproximaciones a normalidad
ax_qq = axes[7]
for clave, color in [('log', PALETA['gold']),
                     ('boxcox', PALETA['plum']),
                     ('quantile', PALETA['mosslt'])]:
    tr_train, _ = transformaciones[clave]
    osm, osr = stats.probplot(tr_train, dist='norm', fit=False)
    ax_qq.scatter(osm, osr, alpha=0.4, s=8, color=color, label=nombres_pretty[clave].split()[0])
ax_qq.plot([-3, 3], [-3, 3], 'k--', alpha=0.5, lw=1, label='Normal teorica')
ax_qq.set_title('Q-Q PLOT: Log vs Box-Cox vs Quantile', fontweight='bold')
ax_qq.set_xlabel('Cuantiles teoricos (normal)')
ax_qq.set_ylabel('Cuantiles muestrales')
ax_qq.legend(fontsize=9, loc='upper left')

plt.suptitle(f'Comparativa de transformaciones sobre {variable}',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 📝 Interpretación de la comparativa

**Min-Max y Z-score** preservan la forma de la distribución; solo modifican la escala. Por eso siguen mostrando la misma asimetría que el original. Son útiles para algoritmos sensibles a escala (KNN, SVM con RBF, redes neuronales) pero **no normalizan la forma**.

**Logarítmica, Box-Cox y Yeo-Johnson** sí modifican la forma, comprimiendo la cola derecha. Box-Cox optimiza el parámetro $\lambda$ por máxima verosimilitud para acercar al máximo a la normal. El Q-Q plot revela qué tan bien lo logran: cuanto más cerca de la diagonal punteada, mejor la aproximación.

**Quantile transform** garantiza por construcción una distribución normal estándar exacta —el Q-Q plot es prácticamente lineal—. Pero pierde la relación funcional original con los datos: dos observaciones con el mismo rango percentilar tendrán el mismo valor transformado, aunque sus valores originales sean muy distintos.

---
## 2.3 Tabla decisional: ¿qué transformación usar en cada caso?


In [ ]:
# ══════════════════════════════════════════════════════════════
# 2.4 TABLA DECISIONAL CONSOLIDADA
# ══════════════════════════════════════════════════════════════

decision_table = pd.DataFrame([
    ['Variable monetaria con asimetria positiva',  'log(1+x)',     'Comprime cola derecha, preserva interpretacion'],
    ['Algoritmo basado en distancias (KNN, SVM)',   'z-score o min-max', 'Las distancias se distorsionan sin escalar'],
    ['Regresion lineal con normalidad de errores',  'Box-Cox / Yeo-Johnson', 'Optimiza lambda para aproximar normal'],
    ['Random Forest, XGBoost (arboles)',            'NINGUNA',      'Insensibles a transformaciones monotonas'],
    ['Outliers extremos no tratados',                'Quantile transform', 'Robusta porque solo importan los rangos'],
    ['Red neuronal con activacion sigmoide/tanh',   'Min-max [0,1] o [-1,1]', 'Evita saturacion de gradientes'],
    ['Variables en porcentaje o probabilidad',      'NINGUNA o logit', 'Ya estan acotadas en [0,1]'],
    ['Score crediticio para regulacion',             'log + WoE binning', 'Estandar bancario, interpretable']
], columns=['Escenario', 'Transformacion', 'Justificacion'])

print('TABLA DECISIONAL DE TRANSFORMACIONES NUMERICAS')
print('=' * 100)
print(decision_table.to_string(index=False))
print('=' * 100)

### 📝 La excepción de los árboles

Es contraintuitivo pero crucial: **los algoritmos basados en árboles (Random Forest, XGBoost, LightGBM) son insensibles a transformaciones monotónicas**. Un árbol que decide a partir de `credit_amount > 5000` toma exactamente la misma decisión que uno que decide `log(1 + credit_amount) > log(5001)` —porque la transformación logarítmica es estrictamente creciente y no altera el orden relativo de las observaciones.

Por lo tanto, transformar variables antes de entrenar un Random Forest es trabajo perdido y, peor aún, dificulta la interpretación de la importancia de variables. La regla práctica: si el modelo final será de árbol, dejar las variables en su escala natural.

---
## 2.4 Aplicación al pipeline: transformación de las features numéricas


In [ ]:
# ══════════════════════════════════════════════════════════════
# 2.5 APLICACION SISTEMATICA AL DATASET DE TRABAJO
# ══════════════════════════════════════════════════════════════

# Variables numericas que requieren transformacion (monetarias, asimetricas)
vars_log = ['credit_amount', 'monthly_payment', 'credit_per_emp_year', 'amount_per_age',
            'emp_x_amount', 'age_x_duration']

# Aplicamos log1p (transformacion sin parametros aprendidos)
for v in vars_log:
    X_train_fe[f'{v}_log'] = np.log1p(X_train_fe[v])
    X_test_fe[f'{v}_log']  = np.log1p(X_test_fe[v])

print(f'Aplicada transformacion log(1+x) a {len(vars_log)} variables monetarias.')

# Variables que requieren estandarizacion (para el modelo lineal regularizado)
vars_zscore = ['age', 'duration', 'employment_num', 'liquidity_score']
scaler_z = StandardScaler()
X_train_fe[[f'{v}_z' for v in vars_zscore]] = scaler_z.fit_transform(X_train_fe[vars_zscore])
X_test_fe[[f'{v}_z' for v in vars_zscore]]  = scaler_z.transform(X_test_fe[vars_zscore])

print(f'Aplicada estandarizacion z-score a {len(vars_zscore)} variables.')

# Verificacion: las transformadas en TRAIN tienen media ~ 0 y desvio ~ 1
print('\nVerificacion z-score en TRAIN (deben ser ~0 y ~1):')
for v in vars_zscore:
    m = X_train_fe[f'{v}_z'].mean()
    s = X_train_fe[f'{v}_z'].std()
    print(f'  {v}_z: media = {m:>+.6f}  desvio = {s:.4f}')

# Verificacion: las transformadas en TEST NO tienen media exactamente 0 y desvio 1
# (eso es correcto: usamos los parametros del train)
print('\nVerificacion z-score en TEST (NO deben ser exactamente 0 y 1):')
for v in vars_zscore:
    m = X_test_fe[f'{v}_z'].mean()
    s = X_test_fe[f'{v}_z'].std()
    print(f'  {v}_z: media = {m:>+.6f}  desvio = {s:.4f}')

print('\nLa diferencia entre las medias/desvios de train y test es PROPIA del')
print('proceso correcto: aplicamos los parametros aprendidos en train al test.')

---
# 🔤 BLOQUE 3 — Codificación de variables categóricas

Las variables categóricas no pueden alimentarse directamente a la mayoría de los algoritmos: requieren conversión a representación numérica. La elección del codificador afecta materialmente la performance y, sobre todo, condiciona la posibilidad de fuga del objetivo (target leakage).

Recorreremos los **seis codificadores canónicos** sobre las variables categóricas del German Credit, con énfasis especial en el **target encoding regularizado** —el más potente y simultáneamente el más riesgoso—.

| Codificador | Cuando usar | Cuidados |
|:---|:---|:---|
| **One-Hot** | Modelos lineales, baja cardinalidad | Explosión de dimensionalidad |
| **Label** | Árboles, sin orden inherente | No usar en modelos lineales |
| **Ordinal** | Cualquier modelo, orden conocido | Requiere expertise de dominio |
| **Target encoding** | Alta cardinalidad, modelos cualquiera | **Regularización OBLIGATORIA** |
| **Binary** | Cardinalidad alta, alternativa a one-hot | Menos interpretable |
| **Frequency** | Exploración rápida, baseline | Pierde identidad de la categoría |

---
## 3.1 Inventario de variables categóricas y cardinalidad


In [ ]:
# ══════════════════════════════════════════════════════════════
# 3.1 INVENTARIO DE VARIABLES CATEGORICAS
# ══════════════════════════════════════════════════════════════

cat_vars = X_train_fe.select_dtypes(include=['category', 'object']).columns.tolist()
# Filtramos las que son derivadas de Bloque 1 y son ya manejables
cat_vars = [v for v in cat_vars if v in X_train.columns]  # solo las originales

cardinalidad = pd.DataFrame({
    'variable': cat_vars,
    'n_unicos': [X_train_fe[v].nunique() for v in cat_vars],
    'frecuencia_top1': [X_train_fe[v].value_counts(normalize=True).iloc[0] for v in cat_vars]
}).sort_values('n_unicos', ascending=False).reset_index(drop=True)

# Clasificamos por cardinalidad
def clasificar_card(n):
    if n <= 3: return 'Baja'
    elif n <= 6: return 'Media'
    else: return 'Alta'

cardinalidad['nivel'] = cardinalidad['n_unicos'].apply(clasificar_card)
cardinalidad['recomendacion'] = cardinalidad['nivel'].map({
    'Baja':  'One-Hot o Ordinal',
    'Media': 'One-Hot, Ordinal o Target encoding',
    'Alta':  'Target encoding regularizado o Binary'
})

print('CARDINALIDAD DE VARIABLES CATEGORICAS')
print('=' * 80)
print(cardinalidad.to_string(index=False))
print('=' * 80)

### 📝 Interpretación

La cardinalidad determina el codificador apropiado. Variables con pocos niveles (3-6) admiten one-hot sin problema dimensional. Variables con cardinalidad media (7-12) ya empiezan a generar matrices dispersas grandes; ahí target encoding regularizado o binary encoding son alternativas más eficientes. Variables con cardinalidad alta (>12) descartan one-hot por completo.

En el German Credit, la mayor cardinalidad la alcanzan `purpose` (10 niveles) y `personal_status` (4 niveles); el resto está en cardinalidad baja a media. Es un dataset relativamente cómodo desde este punto de vista —en datasets reales con códigos postales o categorías comerciales, la cardinalidad puede alcanzar miles—.

---
## 3.2 One-Hot Encoding


In [ ]:
# ══════════════════════════════════════════════════════════════
# 3.2 ONE-HOT ENCODING
# ══════════════════════════════════════════════════════════════

# Aplicamos one-hot a las variables de cardinalidad baja-media
vars_onehot = ['purpose', 'housing', 'job', 'personal_status', 'foreign_worker']

# Usamos get_dummies con drop_first=True para evitar la trampa de la dummy variable
X_train_oh = pd.get_dummies(X_train_fe[vars_onehot], drop_first=True, dtype=int)
X_test_oh  = pd.get_dummies(X_test_fe[vars_onehot],  drop_first=True, dtype=int)

# Alineamos columnas (puede haber categorias en train que no esten en test)
X_test_oh = X_test_oh.reindex(columns=X_train_oh.columns, fill_value=0)

print(f'One-Hot aplicado a {len(vars_onehot)} variables:')
print(f'  Variables originales: {len(vars_onehot)}')
print(f'  Columnas resultantes: {X_train_oh.shape[1]}')
print(f'  Factor de expansion: {X_train_oh.shape[1] / len(vars_onehot):.1f}x')
print()
print('Primeras 5 columnas one-hot:')
print(X_train_oh.iloc[:, :5].head())

### 📝 La trampa de la dummy variable

Al aplicar one-hot, **debemos eliminar una categoría de referencia** (parámetro `drop_first=True`). Si dejamos las K categorías codificadas con K columnas, las columnas son linealmente dependientes —su suma es siempre 1—, lo cual produce multicolinealidad perfecta y rompe la regresión lineal/logística.

Los modelos basados en árboles toleran esta redundancia, pero la práctica habitual es siempre dropear la primera por consistencia.

---
## 3.3 Label Encoding y Ordinal Encoding


In [ ]:
# ══════════════════════════════════════════════════════════════
# 3.3 LABEL ENCODING vs ORDINAL ENCODING
# ══════════════════════════════════════════════════════════════

# (a) Label encoding: orden alfabetico (arbitrario)
le = LabelEncoder()
X_train_fe['purpose_label'] = le.fit_transform(X_train_fe['purpose'].astype(str))
# Para test, mapear con el mismo le (atencion a categorias nuevas)
mapping_le = dict(zip(le.classes_, range(len(le.classes_))))
X_test_fe['purpose_label'] = X_test_fe['purpose'].astype(str).map(mapping_le).fillna(-1).astype(int)

print('Label encoding de purpose (orden alfabetico, arbitrario):')
print('-' * 50)
for cat, code_v in sorted(mapping_le.items(), key=lambda x: x[1]):
    print(f'  {cat:<25} -> {code_v}')

# (b) Ordinal encoding: orden definido por dominio
# La variable savings_status tiene un orden natural: a mayor saldo, menor riesgo
savings_orden = ['no known savings', '<100', '100<=X<500', '500<=X<1000', '>=1000']
oe = OrdinalEncoder(categories=[savings_orden])
X_train_fe['savings_oe'] = oe.fit_transform(X_train_fe[['savings_status']]).ravel().astype(int)
X_test_fe['savings_oe']  = oe.transform(X_test_fe[['savings_status']]).ravel().astype(int)

print('\n\nOrdinal encoding de savings_status (orden por dominio):')
print('-' * 50)
for i, cat in enumerate(savings_orden):
    n = (X_train_fe['savings_status'] == cat).sum()
    tasa = y_train[X_train_fe['savings_status'] == cat].mean() if n > 0 else np.nan
    print(f'  {i} -> {cat:<22} (n={n:>3}, tasa default = {tasa:.1%})')

print('\nLa monotonicidad de la tasa de default confirma el orden definido.')

### 📝 Interpretación

La diferencia entre **label** y **ordinal** es fundamental. Label encoding asigna enteros arbitrariamente —típicamente por orden alfabético—, lo cual es **un error grave en modelos lineales** porque introduce un orden ficticio: el modelo interpretará que `business=0` está "antes" que `car=1`, sin que esto tenga sentido económico.

Ordinal encoding, en cambio, asigna enteros respetando un orden real definido por el dominio. La verificación de monotonicidad —tasa de default decreciente a medida que aumenta el código— valida que el orden propuesto efectivamente captura la jerarquía económica subyacente.

> **Regla práctica.** Label encoding solo es seguro en árboles y bosques, donde el orden numérico no se interpreta linealmente. En regresión lineal/logística, **siempre** ordinal o one-hot.

---
## 3.4 Target Encoding NAÏVE — el problema que motiva la regularización

Antes de mostrar la versión correcta, exhibamos el problema. La codificación naïve consiste en reemplazar cada categoría por la media simple del target condicionada a esa categoría, calculada sobre todo el train.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 3.4 TARGET ENCODING NAIVE - DEMOSTRACION DEL PROBLEMA
# ══════════════════════════════════════════════════════════════

# Calculo de la media del target por categoria de purpose (NAIVE)
target_naive = y_train.groupby(X_train_fe['purpose']).agg(['mean', 'count']).reset_index()
target_naive.columns = ['purpose', 'tasa_default', 'n_obs']
target_naive['tasa_default'] = target_naive['tasa_default'].round(4)
target_naive = target_naive.sort_values('n_obs', ascending=False)

print('Target encoding NAIVE por categoria de purpose:')
print('=' * 60)
print(target_naive.to_string(index=False))
print('=' * 60)

# El problema: las categorias raras tienen estimaciones inestables
print('\nObservaciones criticas:')
print('-' * 60)
raras = target_naive[target_naive['n_obs'] < 30]
abundantes = target_naive[target_naive['n_obs'] >= 100]
print(f'Categorias con n < 30:  {len(raras)} categorias')
for _, row in raras.iterrows():
    print(f'  - {row["purpose"]:<15} n = {row["n_obs"]:>3}  tasa = {row["tasa_default"]:.1%}')
print(f'\nCategorias con n >= 100: {len(abundantes)} categorias')
for _, row in abundantes.iterrows():
    print(f'  - {row["purpose"]:<15} n = {row["n_obs"]:>3}  tasa = {row["tasa_default"]:.1%}')

print('\nEl problema: las categorias con pocas observaciones producen estimaciones')
print('volatiles y propensas a overfitting. Necesitamos regularizar hacia la media global.')

---
## 3.5 Target Encoding REGULARIZADO — fórmula bayesiana de suavizado

La solución al problema de las categorías raras es la **fórmula de suavizado bayesiano**:

$$\mu_c = \frac{n_c \cdot \bar{y}_c + m \cdot \bar{y}_{global}}{n_c + m}$$

Donde:
- $n_c$ = número de observaciones de la categoría $c$
- $\bar{y}_c$ = media local del target en la categoría $c$
- $\bar{y}_{global}$ = media global del target sobre todo el train
- $m$ = peso del prior (típicamente 10–100)

**Intuición bayesiana:** estamos calculando una media ponderada entre la media local y la media global. Cuando $n_c$ es grande, el peso del prior se vuelve insignificante. Cuando $n_c$ es pequeño, el prior global domina. En el caso extremo $n_c = 0$ (categoría nueva en test), el resultado es exactamente la media global.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 3.5 TARGET ENCODING REGULARIZADO - implementacion manual y libreria
# ══════════════════════════════════════════════════════════════

# (A) IMPLEMENTACION MANUAL para entender la formula
def target_encode_smooth(serie_train, target_train, serie_test, m=10):
    """
    Target encoding regularizado con suavizado bayesiano.
    
    Parametros:
        serie_train : pd.Series con la variable categorica del train
        target_train : pd.Series con el target binario del train
        serie_test : pd.Series con la misma variable en test
        m : int, peso del prior (default 10)
    
    Retorna:
        encoded_train, encoded_test
    """
    media_global = target_train.mean()
    stats_cat = target_train.groupby(serie_train).agg(['mean', 'count'])
    stats_cat.columns = ['mean_local', 'n']
    
    # Formula bayesiana
    stats_cat['encoded'] = (
        (stats_cat['n'] * stats_cat['mean_local'] + m * media_global) /
        (stats_cat['n'] + m)
    )
    
    # Mapeo
    mapping = stats_cat['encoded'].to_dict()
    encoded_train = serie_train.map(mapping)
    encoded_test  = serie_test.map(mapping).fillna(media_global)  # categorias nuevas -> global
    
    return encoded_train, encoded_test, stats_cat

# Aplicacion sobre purpose
enc_train, enc_test, stats_purpose = target_encode_smooth(
    X_train_fe['purpose'], y_train, X_test_fe['purpose'], m=10
)
X_train_fe['purpose_te'] = enc_train
X_test_fe['purpose_te']  = enc_test

print('Comparacion NAIVE vs REGULARIZADO (m=10) para purpose:')
print('=' * 75)
print(f'Media global: {y_train.mean():.4f}')
print('-' * 75)
print(f'{"Categoria":<18} {"n":<5} {"Naive":<10} {"Regulado":<10} {"Cambio":<10}')
print('-' * 75)
for cat in stats_purpose.index:
    n = int(stats_purpose.loc[cat, 'n'])
    naive = stats_purpose.loc[cat, 'mean_local']
    reg = stats_purpose.loc[cat, 'encoded']
    cambio = reg - naive
    direccion = '<-' if cambio > 0 else '->' if cambio < 0 else '-'
    print(f'{cat:<18} {n:<5} {naive:<10.4f} {reg:<10.4f} {cambio:>+8.4f} {direccion}')
print('=' * 75)
print('\nLectura: las categorias con pocas observaciones se desplazan mas hacia')
print('la media global (suavizado fuerte). Las categorias con muchas observaciones')
print('preservan su tasa empirica (suavizado debil).')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 3.5.b VERSION CON LIBRERIA category_encoders
# ══════════════════════════════════════════════════════════════

# La libreria implementa el suavizado correctamente y maneja CV internos
te_enc = ce.TargetEncoder(cols=['purpose', 'job', 'housing'], smoothing=10)

# Aplicacion respetando train/test
X_train_te = te_enc.fit_transform(X_train_fe[['purpose', 'job', 'housing']], y_train)
X_test_te  = te_enc.transform(X_test_fe[['purpose', 'job', 'housing']])

X_train_te.columns = [f'{c}_te' for c in X_train_te.columns]
X_test_te.columns  = [f'{c}_te' for c in X_test_te.columns]

print(f'Aplicado target encoding regularizado a 3 variables:')
print(f'  Forma X_train_te: {X_train_te.shape}')
print(f'  Forma X_test_te:  {X_test_te.shape}')
print()
print('Primeras 5 filas de X_train_te:')
print(X_train_te.head().round(4))
print()
print('Distribucion de cada variable codificada:')
print(X_train_te.describe().round(4))

### ⚠️ Advertencia operativa: target encoding y validación cruzada

El código anterior, aunque didáctico, comete una sutileza problemática en producción: el target encoding se calculó usando **todo el training set** y luego se aplicó al mismo training para entrenar el modelo. Esto significa que cada observación contribuye a su propia codificación —técnicamente, una forma leve de fuga.

**La práctica rigurosa** es calcular el encoding dentro de un esquema de cross-validation: para cada fold, computar el encoding usando solo las observaciones de los demás folds. La librería `category_encoders` tiene una clase `LeaveOneOutEncoder` y `CatBoostEncoder` que implementan estas estrategias. En proyectos profesionales con regulación bancaria, encapsular el target encoding dentro de un `Pipeline` con cross-validation es obligatorio.

---
## 3.6 Frequency Encoding y Binary Encoding


In [ ]:
# ══════════════════════════════════════════════════════════════
# 3.6 FRECUENCIA Y BINARY ENCODING
# ══════════════════════════════════════════════════════════════

# (A) Frequency encoding sobre purpose
freq_map = X_train_fe['purpose'].value_counts(normalize=True).to_dict()
X_train_fe['purpose_freq'] = X_train_fe['purpose'].astype(str).map(freq_map).astype(float)
X_test_fe['purpose_freq']  = X_test_fe['purpose'].astype(str).map(freq_map).fillna(0).astype(float)

print('Frequency encoding de purpose:')
print('-' * 45)
for cat in sorted(freq_map.keys()):
    f = freq_map[cat]
    print(f'  {cat:<18} -> {f:.4f}')

# (B) Binary encoding (cardinalidad reducida)
be_enc = ce.BinaryEncoder(cols=['purpose'])
X_train_be = be_enc.fit_transform(X_train_fe[['purpose']])
X_test_be  = be_enc.transform(X_test_fe[['purpose']])

print(f'\nBinary encoding de purpose:')
print(f'  Niveles originales:  {X_train_fe["purpose"].nunique()}')
print(f'  Columnas resultantes: {X_train_be.shape[1]}')
print(f'  Reduccion: log2(10) ~= {np.log2(10):.1f} (en lugar de 10 columnas one-hot)')
print()
print('Primeras 5 filas codificadas:')
print(X_train_be.head())

### 📝 Síntesis del Bloque 3

Cada codificador resuelve un problema específico:

- **One-Hot**: estándar para baja cardinalidad y modelos lineales. Drop one para evitar multicolinealidad.
- **Label**: solo para árboles. En modelos lineales introduce orden artificial.
- **Ordinal**: cuando existe orden real en el dominio. Validar con monotonicidad de tasa de default.
- **Target encoding**: máximo poder predictivo, pero **siempre regularizado** con suavizado bayesiano y, idealmente, dentro de un esquema de CV.
- **Binary**: reduce dimensionalidad logarítmicamente, alternativa moderada a one-hot.
- **Frequency**: trivial y a veces efectivo, pero pierde la identidad de la categoría.

> **Regla operativa.** En modelos lineales: ordinal donde haya orden, one-hot en el resto, target encoding regularizado solo si la cardinalidad es alta. En modelos de árboles: label encoding sirve perfectamente y es el más eficiente computacionalmente.


---
# 🎯 BLOQUE 4 — Selección de Variables

> *"More variables make the curse of dimensionality bite harder."* — **Hastie, Tibshirani & Friedman**, *The Elements of Statistical Learning*

La selección de variables es uno de los terrenos más densos conceptualmente del feature engineering. Tres familias metodológicas estructuran el campo:

| Familia | Mecánica | Pros | Contras |
|:---|:---|:---|:---|
| **Filter** | Criterios estadísticos univariantes pre-modelado | Rápido, escalable | Ignora interacciones |
| **Wrapper** | Modelo como caja negra evaluando subconjuntos | Capta interacciones | Costo computacional alto |
| **Embedded** | Selección integrada al entrenamiento | Balance favorable | Atado al algoritmo |

Aplicaremos cada una sobre el dataset enriquecido del German Credit y compararemos los rankings resultantes.

---
## 4.1 Construcción de la matriz de features para selección


In [ ]:
# ══════════════════════════════════════════════════════════════
# 4.1 CONSOLIDACION DE LA MATRIZ DE FEATURES PARA SELECCION
# ══════════════════════════════════════════════════════════════

# Combinamos las features numericas (originales y derivadas) con las codificadas
# Excluimos las que son redundantes (tenemos la version transformada)

# Numericas originales que dejamos como estan
num_originales = ['age', 'duration', 'credit_amount', 'installment_commitment',
                  'residence_since', 'existing_credits', 'num_dependents']

# Numericas derivadas del Bloque 1
num_derivadas = ['monthly_payment', 'credit_per_emp_year', 'amount_per_age',
                  'is_young_borrower', 'stable_employment', 'long_term_loan', 'high_amount',
                  'emp_x_amount', 'age_x_duration', 'young_long_loan',
                  'employment_num', 'checking_ord', 'savings_ord', 'liquidity_score']

# Codificaciones one-hot (ya en X_train_oh)
# Codificaciones target encoding
te_cols = ['purpose_te', 'job_te', 'housing_te']

# Construimos X_train_full y X_test_full
X_train_full = pd.concat([
    X_train_fe[num_originales + num_derivadas].reset_index(drop=True),
    X_train_oh.reset_index(drop=True),
    X_train_te.reset_index(drop=True)
], axis=1)

X_test_full = pd.concat([
    X_test_fe[num_originales + num_derivadas].reset_index(drop=True),
    X_test_oh.reset_index(drop=True),
    X_test_te.reset_index(drop=True)
], axis=1)

# Aseguramos tipos numericos
X_train_full = X_train_full.astype(float)
X_test_full  = X_test_full.astype(float)

y_train_arr = y_train.reset_index(drop=True)
y_test_arr  = y_test.reset_index(drop=True)

print(f'Matriz consolidada para seleccion:')
print(f'  X_train_full: {X_train_full.shape}')
print(f'  X_test_full:  {X_test_full.shape}')
print(f'  Total features candidatas: {X_train_full.shape[1]}')
print(f'  Tasa de default train: {y_train_arr.mean():.1%}')

---
## 4.2 Métodos Filter — criterios estadísticos univariantes

Los filtros evalúan cada variable independientemente del modelo. Son rápidos, escalables y útiles como primer cribado, pero ignoran las interacciones entre variables.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 4.2 FILTER 1: PEARSON Y SPEARMAN
# ══════════════════════════════════════════════════════════════

# Correlacion de Pearson (relacion lineal)
corr_pearson = X_train_full.apply(lambda x: stats.pearsonr(x, y_train_arr)[0]).abs()

# Correlacion de Spearman (relacion monotona, robusta a no linealidad)
corr_spearman = X_train_full.apply(lambda x: stats.spearmanr(x, y_train_arr)[0]).abs()

# Top 15 por cada criterio
filter_corr = pd.DataFrame({
    'feature': X_train_full.columns,
    '|Pearson|': corr_pearson.values,
    '|Spearman|': corr_spearman.values
}).sort_values('|Spearman|', ascending=False).reset_index(drop=True)

print('TOP 15 features por correlacion (Pearson y Spearman):')
print('=' * 65)
print(filter_corr.head(15).to_string(index=False))
print('=' * 65)

In [ ]:
# ══════════════════════════════════════════════════════════════
# 4.3 FILTER 2: INFORMACION MUTUA Y ANOVA F
# ══════════════════════════════════════════════════════════════

# Informacion mutua (capta cualquier dependencia, lineal o no)
mi_scores = mutual_info_classif(X_train_full, y_train_arr, random_state=SEED)

# ANOVA F-test (asume normalidad pero es robusto)
f_scores, f_pvalues = f_classif(X_train_full, y_train_arr)

filter_uni = pd.DataFrame({
    'feature': X_train_full.columns,
    'MI': mi_scores,
    'F_stat': f_scores,
    'F_pvalue': f_pvalues
}).sort_values('MI', ascending=False).reset_index(drop=True)

print('TOP 15 features por Informacion Mutua y ANOVA F:')
print('=' * 75)
print(filter_uni.head(15).round(4).to_string(index=False))
print('=' * 75)
print('\nLectura: MI cercana a 0 = ninguna asociacion. MI alta = fuerte dependencia.')
print('         F_pvalue < 0.05 = la variable difiere significativamente entre clases.')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 4.4 VISUALIZACION COMPARATIVA DE FILTROS
# ══════════════════════════════════════════════════════════════

# Tomamos top 15 por mutual info
top_mi = filter_uni.head(15).copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (a) Mutual info
ax = axes[0]
ax.barh(top_mi['feature'][::-1], top_mi['MI'][::-1], color=PALETA['teal'], edgecolor='white')
ax.set_xlabel('Informacion Mutua')
ax.set_title('TOP 15 por Informacion Mutua', fontweight='bold')
for i, (feat, val) in enumerate(zip(top_mi['feature'][::-1], top_mi['MI'][::-1])):
    ax.text(val + 0.001, i, f'{val:.3f}', va='center', fontsize=9)

# (b) F-statistic (solo para los top mi)
top_mi_f = top_mi.sort_values('F_stat', ascending=False)
ax = axes[1]
ax.barh(top_mi_f['feature'][::-1], top_mi_f['F_stat'][::-1], color=PALETA['plum'], edgecolor='white')
ax.set_xlabel('F-statistic (ANOVA)')
ax.set_title('Top features por F-statistic', fontweight='bold')
for i, (feat, val) in enumerate(zip(top_mi_f['feature'][::-1], top_mi_f['F_stat'][::-1])):
    ax.text(val + 0.5, i, f'{val:.1f}', va='center', fontsize=9)

plt.suptitle('Filtros univariantes — comparacion de rankings', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 📝 Interpretación de los filtros

Los rankings de Mutual Information y F-statistic suelen coincidir en las top features pero pueden divergir en el detalle: F-statistic asume relación lineal mientras que MI capta cualquier dependencia. Cuando una variable aparece alta en MI pero baja en F-statistic, suele indicar relación no lineal con el target.

> **Limitación universal de los filtros.** Una variable que es marginalmente irrelevante puede ser conjuntamente relevante en presencia de otras (interacciones). Los filtros no detectan estas situaciones. Por eso se complementan con métodos wrapper o embedded.

---
## 4.3 Métodos Wrapper — el modelo como juez

Los wrappers usan el modelo final como caja negra para evaluar subconjuntos de variables. Son más precisos pero costosos: cada subconjunto requiere reentrenar el modelo.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 4.5 RFE - Recursive Feature Elimination con CV
# ══════════════════════════════════════════════════════════════

# Estimador base: regresion logistica (rapido y sensible al numero de features)
estimator = LogisticRegression(max_iter=2000, random_state=SEED, solver='liblinear')

# Estandarizamos para que la regularizacion actue equilibradamente
scaler_rfe = StandardScaler()
X_train_scaled = scaler_rfe.fit_transform(X_train_full)
X_test_scaled  = scaler_rfe.transform(X_test_full)

# RFECV: selecciona el numero optimo de variables mediante CV
print('Ejecutando RFECV (puede tomar 1-2 minutos)...')
rfecv = RFECV(
    estimator=estimator,
    step=2,
    min_features_to_select=5,
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    scoring='roc_auc',
    n_jobs=-1
)
rfecv.fit(X_train_scaled, y_train_arr)

print(f'\n[OK] RFECV completado.')
print(f'    Numero optimo de features: {rfecv.n_features_}')
print(f'    AUC en CV con configuracion optima: {rfecv.cv_results_["mean_test_score"].max():.4f}')

# Variables seleccionadas
features_rfe = X_train_full.columns[rfecv.support_].tolist()
print(f'\nFeatures seleccionadas por RFECV ({len(features_rfe)}):')
for f in features_rfe[:20]:  # primeras 20
    print(f'  - {f}')
if len(features_rfe) > 20:
    print(f'  ... y {len(features_rfe) - 20} mas')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 4.5.b VISUALIZACION DEL TRADE-OFF: AUC vs n features
# ══════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(12, 5))

cv_means = rfecv.cv_results_['mean_test_score']
cv_stds  = rfecv.cv_results_['std_test_score']
n_feats  = range(rfecv.min_features_to_select,
                 rfecv.min_features_to_select + len(cv_means) * rfecv.step,
                 rfecv.step)
n_feats = list(n_feats)[:len(cv_means)]

ax.plot(n_feats, cv_means, color=PALETA['plum'], linewidth=2, marker='o', markersize=4)
ax.fill_between(n_feats, cv_means - cv_stds, cv_means + cv_stds,
                color=PALETA['plum'], alpha=0.2)
ax.axvline(rfecv.n_features_, color=PALETA['gold'], linestyle='--', linewidth=2,
           label=f'Optimo: {rfecv.n_features_} features')
ax.set_xlabel('Numero de variables retenidas')
ax.set_ylabel('AUC en cross-validation')
ax.set_title('Trade-off RFECV: numero de features vs performance', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print('Lectura: la curva tipicamente crece, alcanza una meseta y comienza a decaer.')
print('El punto optimo es el inicio de la meseta - mas features no aportan AUC.')

---
## 4.4 Métodos Embedded — Lasso y Random Forest

Los métodos embebidos integran la selección al entrenamiento mismo del modelo. Son los más usados en proyectos productivos.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 4.6 EMBEDDED 1: LASSO con CV (regresion logistica L1)
# ══════════════════════════════════════════════════════════════

# Lasso para clasificacion = LogisticRegression con penalty='l1'
# Buscamos el C optimo (inverso de lambda) por CV
from sklearn.linear_model import LogisticRegressionCV

lasso_cv = LogisticRegressionCV(
    Cs=10,                    # 10 valores logaritmicos de C
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    penalty='l1',
    solver='liblinear',
    scoring='roc_auc',
    max_iter=2000,
    random_state=SEED,
    n_jobs=-1
)
lasso_cv.fit(X_train_scaled, y_train_arr)

print(f'[OK] Lasso CV completado.')
print(f'    C optimo (alpha = 1/C): {lasso_cv.C_[0]:.4f}')
print(f'    AUC promedio en CV:     {lasso_cv.scores_[1].mean():.4f}')

# Coeficientes y seleccion (los no nulos sobreviven)
coef_lasso = pd.DataFrame({
    'feature': X_train_full.columns,
    'coef_lasso': lasso_cv.coef_[0],
    'abs_coef': np.abs(lasso_cv.coef_[0])
}).sort_values('abs_coef', ascending=False).reset_index(drop=True)

n_seleccionadas_lasso = (coef_lasso['coef_lasso'] != 0).sum()
features_lasso = coef_lasso[coef_lasso['coef_lasso'] != 0]['feature'].tolist()

print(f'\nLasso retuvo {n_seleccionadas_lasso} de {X_train_full.shape[1]} features.')
print('\nTop 15 features por |coeficiente Lasso|:')
print('-' * 60)
print(coef_lasso.head(15).round(4).to_string(index=False))

In [ ]:
# ══════════════════════════════════════════════════════════════
# 4.7 EMBEDDED 2: Random Forest - importancia MDI y permutacion
# ══════════════════════════════════════════════════════════════

# Random Forest baseline
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=10,
    random_state=SEED,
    n_jobs=-1
)
rf.fit(X_train_full, y_train_arr)

# (a) MDI - Mean Decrease in Impurity
imp_mdi = pd.DataFrame({
    'feature': X_train_full.columns,
    'imp_MDI': rf.feature_importances_
}).sort_values('imp_MDI', ascending=False).reset_index(drop=True)

# (b) Permutation importance (mas robusta, mas costosa)
print('Calculando permutation importance (puede tomar 30-60 segundos)...')
perm_result = permutation_importance(
    rf, X_test_full, y_test_arr,
    n_repeats=10,
    random_state=SEED,
    n_jobs=-1,
    scoring='roc_auc'
)

imp_perm = pd.DataFrame({
    'feature': X_train_full.columns,
    'imp_perm': perm_result.importances_mean,
    'imp_perm_std': perm_result.importances_std
}).sort_values('imp_perm', ascending=False).reset_index(drop=True)

# Combinacion
imp_rf = imp_mdi.merge(imp_perm, on='feature')

print('\nTOP 15 features Random Forest (MDI vs Permutacion):')
print('=' * 70)
print(imp_rf.head(15).round(4).to_string(index=False))
print('=' * 70)

In [ ]:
# ══════════════════════════════════════════════════════════════
# 4.7.b VISUALIZACION COMPARATIVA: MDI vs Permutation Importance
# ══════════════════════════════════════════════════════════════

top_n = 15
top_imp = imp_rf.head(top_n).copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# MDI
ax = axes[0]
ax.barh(top_imp['feature'][::-1], top_imp['imp_MDI'][::-1],
        color=PALETA['moss'], edgecolor='white')
ax.set_xlabel('Importancia MDI')
ax.set_title('TOP 15 - Importancia MDI (Mean Decrease Impurity)', fontweight='bold')

# Permutation
ax = axes[1]
top_imp_perm = top_imp.sort_values('imp_perm', ascending=False)
ax.barh(top_imp_perm['feature'][::-1], top_imp_perm['imp_perm'][::-1],
        xerr=top_imp_perm['imp_perm_std'][::-1],
        color=PALETA['gold'], edgecolor='white', error_kw={'alpha': 0.5})
ax.set_xlabel('Caida de AUC al permutar')
ax.set_title('TOP 15 - Permutation Importance (con error std)', fontweight='bold')

plt.suptitle('Random Forest — comparacion de medidas de importancia',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\nNota teorica: MDI sufre sesgo hacia variables de alta cardinalidad.')
print('Permutation importance es mas robusta pero mas costosa.')
print('La verdad esta en la interseccion de ambas medidas.')

---
## 4.5 Síntesis comparativa: rankings de los tres paradigmas


In [ ]:
# ══════════════════════════════════════════════════════════════
# 4.8 RANKING COMPARATIVO DE LOS TRES PARADIGMAS
# ══════════════════════════════════════════════════════════════

# Construimos rankings ordinales para cada metodo
def rank_features(df, score_col, n_features):
    """Asigna rango 1 a la mejor feature."""
    df_sorted = df.sort_values(score_col, ascending=False).reset_index(drop=True)
    return dict(zip(df_sorted['feature'], range(1, len(df_sorted) + 1)))

n = X_train_full.shape[1]
rank_mi = rank_features(filter_uni, 'MI', n)
rank_rfe = {f: i+1 for i, f in enumerate(features_rfe)}  # solo selecciona, no ordena
rank_lasso = rank_features(coef_lasso[coef_lasso['coef_lasso']!=0],
                            'abs_coef', n_seleccionadas_lasso)
rank_mdi = rank_features(imp_mdi, 'imp_MDI', n)
rank_perm = rank_features(imp_perm, 'imp_perm', n)

# Tabla comparativa
ranking = pd.DataFrame({'feature': X_train_full.columns})
ranking['filter_MI'] = ranking['feature'].map(rank_mi)
ranking['wrapper_RFE'] = ranking['feature'].map(rank_rfe).fillna(99).astype(int)
ranking['embedded_Lasso'] = ranking['feature'].map(rank_lasso).fillna(99).astype(int)
ranking['embedded_MDI'] = ranking['feature'].map(rank_mdi)
ranking['embedded_Perm'] = ranking['feature'].map(rank_perm)

# Promedio de rangos (excluyendo el 99 placeholder)
def rank_promedio(row):
    valores = [v for v in [row['filter_MI'], row['wrapper_RFE'],
                            row['embedded_Lasso'], row['embedded_MDI'], row['embedded_Perm']]
               if v != 99]
    return np.mean(valores) if valores else 999

ranking['rank_promedio'] = ranking.apply(rank_promedio, axis=1)
ranking = ranking.sort_values('rank_promedio').reset_index(drop=True)

print('RANKING COMPARATIVO - TOP 20 features (rango menor = mejor)')
print('=' * 95)
print(ranking.head(20).to_string(index=False))
print('=' * 95)
print('\nNota: 99 indica que el metodo no selecciono la variable.')

### 📝 Síntesis del Bloque 4

El ranking comparativo revela qué variables son **robustamente** importantes —aquellas que aparecen en el top de los tres paradigmas. Esas son las candidatas firmes para el modelo final. Las que aparecen en uno o dos paradigmas requieren análisis adicional: pueden ser relevantes en interacciones (capturadas por wrapper/embedded pero no por filter univariado) o falsamente importantes por sesgos del método.

> **Regla operativa.** En proyectos profesionales, la práctica robusta combina los tres paradigmas: filter para el cribado inicial, wrapper o embedded para el ranking refinado, y la **intersección de los rankings** define el conjunto final. Variables seleccionadas por al menos dos de los tres paradigmas tienen alta confianza.


---
# ⚠️ BLOQUE 5 — Multicolinealidad y VIF

> *"Cuando dos variables dicen lo mismo, una de ellas miente al modelo lineal."*

El **Variance Inflation Factor** cuantifica cuánto se infla la varianza del coeficiente estimado de cada variable a causa de su correlación con las demás predictoras.

$$\text{VIF}_j = \frac{1}{1 - R^2_j}$$

Donde $R^2_j$ es el coeficiente de determinación de regresar $X_j$ sobre todas las demás predictoras.

**Escala de interpretación:**

| VIF | Diagnóstico | Acción |
|:---:|:---|:---|
| 1 | Sin colinealidad | Ideal (poco frecuente en datos reales) |
| 1–5 | Aceptable | Continuar sin intervención |
| 5–10 | Moderada | Investigar la fuente |
| ≥10 | Severa | Eliminar, combinar o regularizar |

> **Regla regulatoria.** En scoring crediticio bajo Basilea III e IFRS 9, VIF > 5 obliga a justificar técnicamente la inclusión de la variable o a reformular el modelo.

---
## 5.1 Cálculo del VIF inicial sobre el subconjunto seleccionado por Lasso

In [ ]:
# ══════════════════════════════════════════════════════════════
# 5.1 CALCULO DEL VIF SOBRE LAS FEATURES RETENIDAS POR LASSO
# ══════════════════════════════════════════════════════════════

# Trabajamos sobre las features que sobrevivieron al Lasso
X_vif = X_train_scaled[:, [list(X_train_full.columns).index(f) for f in features_lasso]]
X_vif_df = pd.DataFrame(X_vif, columns=features_lasso)

def calcular_vif(X):
    """Calcula el VIF para cada columna del DataFrame X."""
    vif_data = pd.DataFrame({
        'feature': X.columns,
        'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    })
    return vif_data.sort_values('VIF', ascending=False).reset_index(drop=True)

vif_inicial = calcular_vif(X_vif_df)

print(f'VIF inicial sobre {len(features_lasso)} features seleccionadas por Lasso:')
print('=' * 65)
print(vif_inicial.round(2).to_string(index=False))
print('=' * 65)

n_severo = (vif_inicial['VIF'] >= 10).sum()
n_moderado = ((vif_inicial['VIF'] >= 5) & (vif_inicial['VIF'] < 10)).sum()
n_aceptable = (vif_inicial['VIF'] < 5).sum()

print(f'\nResumen del diagnostico inicial:')
print(f'  Severo  (VIF >= 10): {n_severo} variables')
print(f'  Moderado (VIF 5-10): {n_moderado} variables')
print(f'  Aceptable (VIF < 5): {n_aceptable} variables')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 5.2 ELIMINACION ITERATIVA DE VARIABLES CON VIF ALTO
# ══════════════════════════════════════════════════════════════

def eliminar_vif_iterativo(X, umbral=10, verbose=True):
    """
    Elimina iterativamente la variable con mayor VIF mientras supere el umbral.
    """
    X_iter = X.copy()
    eliminadas = []
    iteracion = 0

    while True:
        vif_actual = calcular_vif(X_iter)
        max_vif = vif_actual['VIF'].iloc[0]
        peor_var = vif_actual['feature'].iloc[0]

        if max_vif < umbral:
            if verbose:
                print(f'\n[STOP] Todos los VIF estan por debajo de {umbral}')
            break

        iteracion += 1
        if verbose:
            print(f'Iteracion {iteracion}: eliminar "{peor_var}" (VIF = {max_vif:.2f})')

        eliminadas.append(peor_var)
        X_iter = X_iter.drop(columns=[peor_var])

        if X_iter.shape[1] < 2:
            break

    return X_iter, eliminadas, vif_actual

# Aplicacion con umbral estandar (VIF < 10)
print('Eliminacion iterativa de VIF (umbral = 10):')
print('-' * 60)
X_vif_clean, vars_eliminadas, vif_final = eliminar_vif_iterativo(X_vif_df, umbral=10)

print('\nResumen del proceso (umbral 10):')
print(f'  Variables originales: {X_vif_df.shape[1]}')
print(f'  Variables eliminadas: {len(vars_eliminadas)}')
print(f'  Variables finales:    {X_vif_clean.shape[1]}')
print()
print('VIF final (todos < 10):')
print(vif_final.round(2).to_string(index=False))

In [ ]:
# ══════════════════════════════════════════════════════════════
# 5.3 PRUEBA CON UMBRAL ESTRICTO (REGULATORIO)
# ══════════════════════════════════════════════════════════════

# En contextos regulatorios estrictos se usa VIF < 5
print('Eliminacion iterativa con umbral regulatorio estricto (VIF < 5):')
print('-' * 65)
X_vif_strict, vars_eliminadas_strict, vif_final_strict = eliminar_vif_iterativo(
    X_vif_df, umbral=5
)

print(f'\n  Variables originales: {X_vif_df.shape[1]}')
print(f'  Variables eliminadas: {len(vars_eliminadas_strict)}')
print(f'  Variables finales:    {X_vif_strict.shape[1]}')

# Comparativa visual
fig, ax = plt.subplots(figsize=(12, 5))
escenarios = ['Lasso\n(sin VIF)', 'Lasso + VIF<10', 'Lasso + VIF<5\n(regulatorio)']
n_features = [X_vif_df.shape[1], X_vif_clean.shape[1], X_vif_strict.shape[1]]
bars = ax.bar(escenarios, n_features,
              color=[PALETA['rust'], PALETA['gold'], PALETA['moss']],
              edgecolor='white', linewidth=2)
for bar, n in zip(bars, n_features):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(n), ha='center', fontsize=14, fontweight='bold')
ax.set_ylabel('Numero de features')
ax.set_title('Reduccion progresiva de variables tras filtro VIF', fontweight='bold')
ax.set_ylim(0, max(n_features) * 1.15)
plt.tight_layout()
plt.show()

---
# 🚀 BLOQUE 6 — Pipeline integrado y validación final

Cerramos el notebook con la prueba empírica del pipeline completo. Compararemos **tres niveles de complejidad** para entender el trade-off entre cantidad de variables, performance predictiva y compatibilidad regulatoria:

| Modelo | Features | Filosofía |
|:---|:---:|:---|
| **A — Baseline** | Todas las crudas + one-hot | Mínimo preprocesamiento, sin engineering |
| **B — Enriquecido (Lasso)** | Lasso L1 sobre el set extendido | Engineering + selección embedded |
| **C — Compacto (Lasso + VIF)** | Subconjunto Lasso con VIF < 10 | Apto para scoring regulatorio |

Esta comparación de tres puntos refleja decisiones reales que toma un analista profesional según el contexto: predicción pura (B), modelo bancario auditable (C), o referencia mínima (A).

---
## 6.1 Modelo A — Baseline con variables crudas

In [ ]:
# ══════════════════════════════════════════════════════════════
# 6.1 MODELO A: BASELINE - variables crudas + one-hot total
# ══════════════════════════════════════════════════════════════

# Numericas originales (sin features derivadas) + one-hot de todas las categoricas
vars_numericas_crudas = ['age', 'duration', 'credit_amount', 'installment_commitment',
                          'residence_since', 'existing_credits', 'num_dependents']

todas_cat = X_train.select_dtypes(include=['category', 'object']).columns.tolist()
X_train_baseline_cat = pd.get_dummies(X_train[todas_cat], drop_first=True, dtype=int)
X_test_baseline_cat  = pd.get_dummies(X_test[todas_cat],  drop_first=True, dtype=int)
X_test_baseline_cat = X_test_baseline_cat.reindex(columns=X_train_baseline_cat.columns, fill_value=0)

X_train_baseline = pd.concat([
    X_train[vars_numericas_crudas].reset_index(drop=True),
    X_train_baseline_cat.reset_index(drop=True)
], axis=1).astype(float)

X_test_baseline = pd.concat([
    X_test[vars_numericas_crudas].reset_index(drop=True),
    X_test_baseline_cat.reset_index(drop=True)
], axis=1).astype(float)

scaler_base = StandardScaler()
X_train_base_sc = scaler_base.fit_transform(X_train_baseline)
X_test_base_sc  = scaler_base.transform(X_test_baseline)

cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
model_A = LogisticRegression(penalty='l2', C=1.0, max_iter=2000,
                              random_state=SEED, solver='liblinear')
model_A.fit(X_train_base_sc, y_train_arr)

proba_A_test = model_A.predict_proba(X_test_base_sc)[:, 1]
auc_A_test = roc_auc_score(y_test_arr, proba_A_test)
auc_A_cv = cross_val_score(model_A, X_train_base_sc, y_train_arr,
                            cv=cv, scoring='roc_auc').mean()

print('MODELO A — BASELINE (variables crudas)')
print('=' * 60)
print(f'  Features: {X_train_baseline.shape[1]}')
print(f'  AUC en CV (5-fold) : {auc_A_cv:.4f}')
print(f'  AUC en TEST        : {auc_A_test:.4f}')
print('=' * 60)

In [ ]:
# ══════════════════════════════════════════════════════════════
# 6.2 MODELO B: ENRIQUECIDO — features de Lasso sobre el set extendido
# ══════════════════════════════════════════════════════════════

# features_lasso ya está definido en el bloque 4: las que Lasso retuvo
print(f'Features retenidas por Lasso: {len(features_lasso)}')

X_train_B = X_train_full[features_lasso].values
X_test_B  = X_test_full[features_lasso].values

scaler_B = StandardScaler()
X_train_B_sc = scaler_B.fit_transform(X_train_B)
X_test_B_sc  = scaler_B.transform(X_test_B)

model_B = LogisticRegression(penalty='l2', C=1.0, max_iter=2000,
                              random_state=SEED, solver='liblinear')
model_B.fit(X_train_B_sc, y_train_arr)

proba_B_test = model_B.predict_proba(X_test_B_sc)[:, 1]
auc_B_test = roc_auc_score(y_test_arr, proba_B_test)
auc_B_cv = cross_val_score(model_B, X_train_B_sc, y_train_arr,
                            cv=cv, scoring='roc_auc').mean()

print()
print('MODELO B — ENRIQUECIDO (engineering + Lasso)')
print('=' * 60)
print(f'  Features: {X_train_B.shape[1]}')
print(f'  AUC en CV (5-fold) : {auc_B_cv:.4f}')
print(f'  AUC en TEST        : {auc_B_test:.4f}')
print('=' * 60)

In [ ]:
# ══════════════════════════════════════════════════════════════
# 6.3 MODELO C: COMPACTO — Lasso + VIF < 10 (apto regulatorio)
# ══════════════════════════════════════════════════════════════

features_compacto = X_vif_clean.columns.tolist()
print(f'Features tras Lasso + VIF<10: {len(features_compacto)}')

X_train_C = X_train_full[features_compacto].values
X_test_C  = X_test_full[features_compacto].values

scaler_C = StandardScaler()
X_train_C_sc = scaler_C.fit_transform(X_train_C)
X_test_C_sc  = scaler_C.transform(X_test_C)

model_C = LogisticRegression(penalty='l2', C=1.0, max_iter=2000,
                              random_state=SEED, solver='liblinear')
model_C.fit(X_train_C_sc, y_train_arr)

proba_C_test = model_C.predict_proba(X_test_C_sc)[:, 1]
auc_C_test = roc_auc_score(y_test_arr, proba_C_test)
auc_C_cv = cross_val_score(model_C, X_train_C_sc, y_train_arr,
                            cv=cv, scoring='roc_auc').mean()

print()
print('MODELO C — COMPACTO (Lasso + VIF<10, apto regulatorio)')
print('=' * 60)
print(f'  Features: {X_train_C.shape[1]}')
print(f'  AUC en CV (5-fold) : {auc_C_cv:.4f}')
print(f'  AUC en TEST        : {auc_C_test:.4f}')
print('=' * 60)
print()
print('Variables del modelo compacto:')
for f in features_compacto:
    print(f'  - {f}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 6.4 SINTESIS COMPARATIVA DE LOS TRES MODELOS
# ══════════════════════════════════════════════════════════════

resultados = pd.DataFrame({
    'Modelo': ['A — Baseline', 'B — Enriquecido (Lasso)', 'C — Compacto (Lasso + VIF<10)'],
    'N features': [X_train_baseline.shape[1], X_train_B.shape[1], X_train_C.shape[1]],
    'AUC CV': [auc_A_cv, auc_B_cv, auc_C_cv],
    'AUC TEST': [auc_A_test, auc_B_test, auc_C_test]
})

print('=' * 75)
print('SINTESIS COMPARATIVA DE LOS TRES MODELOS')
print('=' * 75)
print(resultados.round(4).to_string(index=False))
print('=' * 75)

# Visualizacion comparativa
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Panel 1: AUC
ax = axes[0]
modelos = ['A\nBaseline', 'B\nLasso', 'C\nLasso+VIF']
auc_test_vals = [auc_A_test, auc_B_test, auc_C_test]
auc_cv_vals = [auc_A_cv, auc_B_cv, auc_C_cv]
x_pos = np.arange(len(modelos))
width = 0.38
ax.bar(x_pos - width/2, auc_cv_vals, width, label='AUC CV',
       color=PALETA['mosslt'], edgecolor=PALETA['forest'], linewidth=1.5)
ax.bar(x_pos + width/2, auc_test_vals, width, label='AUC TEST',
       color=PALETA['gold'], edgecolor=PALETA['forest'], linewidth=1.5)
for i, (cv_v, test_v) in enumerate(zip(auc_cv_vals, auc_test_vals)):
    ax.text(i - width/2, cv_v + 0.005, f'{cv_v:.3f}', ha='center', fontsize=10, fontweight='bold')
    ax.text(i + width/2, test_v + 0.005, f'{test_v:.3f}', ha='center', fontsize=10, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(modelos)
ax.set_ylabel('AUC')
ax.set_title('Performance: AUC en CV y Test', fontweight='bold')
ax.set_ylim(min(auc_test_vals + auc_cv_vals) - 0.05,
           max(auc_test_vals + auc_cv_vals) + 0.04)
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Panel 2: N features
ax = axes[1]
ns = [X_train_baseline.shape[1], X_train_B.shape[1], X_train_C.shape[1]]
colores_n = [PALETA['rust'], PALETA['plum'], PALETA['moss']]
bars = ax.bar(modelos, ns, color=colores_n, edgecolor=PALETA['forest'], linewidth=1.5)
for bar, n in zip(bars, ns):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(n), ha='center', fontsize=14, fontweight='bold')
ax.set_ylabel('Numero de features')
ax.set_title('Parsimonia: cantidad de variables', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.suptitle('Comparativa de los tres modelos del pipeline',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 📝 Lectura del trade-off

Los tres modelos representan tres puntos diferentes en el espacio de decisiones del analista:

- **Modelo A (Baseline)** suele alcanzar el AUC más alto en absoluto porque conserva toda la información disponible. Sin embargo, no es defendible regulatoriamente: 48 variables sin justificación económica explícita son inauditables ante un comité de validación.

- **Modelo B (Enriquecido con Lasso)** equilibra performance y parsimonia. El Lasso retuvo automáticamente las features con mayor coeficiente, descartando las redundantes. Es el punto más usado en proyectos de modelado predictivo donde la interpretabilidad importa pero no es obligatoria.

- **Modelo C (Compacto, apto regulatorio)** es el modelo que sobreviviría una validación bancaria estricta. Sacrifica algo de AUC a cambio de cumplir VIF < 10 y reducir el modelo a un puñado de variables, todas con justificación económica explícita.

> **Decisión profesional.** En una entidad financiera real, el analista presenta los tres modelos al comité y la decisión final depende del propósito. Para un modelo de marketing predictivo de propensión, el modelo B suele ganar. Para un modelo de scoring crediticio sujeto a regulación BCRA y comité de modelos, el modelo C es la elección obligada aún a costa de algunos puntos de AUC.

### 📝 Nota sobre la magnitud de las diferencias

La magnitud específica de las diferencias entre los tres modelos depende del dataset y del azar muestral. En el German Credit real, las diferencias suelen ser de 1–4 puntos de AUC. En datasets más grandes y con relaciones más complejas (cientos de variables, millones de observaciones), las diferencias pueden ampliarse. La lección que perdura es la **estructura del trade-off**: más variables tiende a mejor AUC pero peor parsimonia, interpretabilidad y robustez regulatoria.

---
# 📌 Conclusiones

### Síntesis cuantitativa

Logramos cuantificar empíricamente el trade-off entre **performance, parsimonia y aptitud regulatoria** en los tres modelos comparados. El feature engineering bien hecho permite reducir drásticamente la cantidad de variables conservando la mayor parte del poder predictivo —una propiedad que se vuelve decisiva en contextos profesionales con requerimientos de interpretabilidad y auditabilidad.

### Síntesis conceptual

Recorrimos los **cinco bloques canónicos** del feature engineering aplicado:

1. **Construcción** de 15 features derivadas en cinco familias (ratios, indicadoras, interacciones, binning, ordinales agregadas).
2. **Transformación** numérica con las seis técnicas (min-max, z-score, log, Box-Cox, Yeo-Johnson, quantile), aplicando la regla de oro fit-en-train.
3. **Codificación** categórica con los seis codificadores, con énfasis especial en el target encoding regularizado mediante suavizado bayesiano.
4. **Selección** de variables aplicando los tres paradigmas (filter, wrapper, embedded) y comparando rankings.
5. **Diagnóstico** de multicolinealidad mediante VIF iterativo, con umbral estándar (10) y regulatorio estricto (5).

### Lecciones operativas para el trabajo profesional

- **El conocimiento del dominio pesa más que el algoritmo.** Ninguna técnica de ML compensa una mala construcción de variables.
- **La regla fit-en-train es absoluta.** Cualquier transformación o codificación que aprenda parámetros del target debe aprenderlos exclusivamente del entrenamiento.
- **El target encoding sin regularización es la fuente más común de overfitting catastrófico.** El suavizado bayesiano y, idealmente, el cálculo dentro de un esquema de cross-validation son obligatorios.
- **La selección robusta combina paradigmas.** Filter para cribado inicial, wrapper o embedded para ranking refinado, intersección como criterio final.
- **El VIF cierra el diagnóstico.** En modelos lineales para scoring crediticio regulatorio, VIF < 5 es el estándar.
- **No hay un modelo único óptimo.** El analista profesional presenta varios modelos y argumenta la elección según el propósito.

### Próxima clase

En la **Clase 5 (jueves 7 de mayo)** entramos en la Unidad 3: *Modelos de Clasificación Aplicados a Finanzas*. Comenzamos con el modelo lineal de referencia para scoring crediticio: la **regresión logística**. Vamos a recorrer los fundamentos matemáticos (función sigmoide, máxima verosimilitud, interpretación vía odds ratio) y a construir nuestra primera **scorecard** sobre el dataset enriquecido que dejamos preparado en este notebook.

---

### 📚 Referencias

- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.). Springer.
- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2013). *An Introduction to Statistical Learning*. Springer.
- Kuhn, M., & Johnson, K. (2019). *Feature Engineering and Selection: A Practical Approach for Predictive Models*. CRC Press.
- Zheng, A., & Casari, A. (2018). *Feature Engineering for Machine Learning*. O'Reilly.
- *category_encoders documentation* — https://contrib.scikit-learn.org/category_encoders/

---

**Dr. Darío Ezequiel Díaz** · drdarioezequieldiaz@gmail.com
*Maestría en Minería de Datos · UTN Facultad Regional Rosario · 2026*